# SIT720 11.1 HD research analysis

This executed notebook examines saved results from `run_research.py`. The full model implementations are in that script.

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd()
assert (ROOT / 'data/heart.csv').exists(), 'Open the notebook from the project folder'
print('Project data and results located.')

Project data and results located.


## Re-running the experiments
Run `python run_research.py` followed by `python analyse_results.py` in the project terminal. These commands fit all models and regenerate tables, predictions and figures. The cells below display those output files; they do not retrain.

In [2]:
df = pd.read_csv(ROOT / 'data/heart.csv')
print('Rows:', len(df))
print('Distinct records:', len(df.drop_duplicates()))
print('Extra duplicates:', int(df.duplicated().sum()))
print('Missing cells:', int(df.isna().sum().sum()))
print('Class counts:')
print(df.target.value_counts().sort_index().to_string())

Rows: 1025
Distinct records: 302
Extra duplicates: 723
Missing cells: 0
Class counts:
target
0    499
1    526


## Published and reproduced results
The random-row diagnostic reproduction allows repeated profiles to cross the split. It is an assumption-based reproduction, not a reliable new-patient estimate.

In [3]:
print('Published results:')
print(pd.read_csv(ROOT / 'results/published_metrics.csv').round(4).to_string(index=False))
print('Measured reproduction:')
print(pd.read_csv(ROOT / 'results/reproduction_metrics.csv').round(4).to_string(index=False))
split = json.loads((ROOT / 'results/reproduction_split.json').read_text())
print('Test rows with matching training profile:', split['overlapping_test_rows'], '/', split['test_rows'])

Published results:
   model  accuracy  precision  recall     f1    auc
      LR    0.8439     0.8196  0.9090 0.8620 0.9204
      DT    0.9268     0.9702  0.8909 0.9289 0.9702
      RF    0.9268     0.9130  0.9545 0.9333 0.9734
     XGB    0.9073     0.9174  0.9090 0.9132 0.9830
      NB    0.8439     0.8250  0.9000 0.8608 0.9185
     KNN    0.8585     0.8648  0.8727 0.8687 0.9304
Stacking    0.9853     1.0000  0.9727 0.9861 0.9880
Measured reproduction:
   model  accuracy  precision  recall     f1    auc  brier  log_loss
      LR    0.8098     0.7619  0.9143 0.8312 0.9298 0.1114    0.3483
      DT    0.9854     1.0000  0.9714 0.9855 0.9857 0.0146    0.5275
      RF    1.0000     1.0000  1.0000 1.0000 1.0000 0.0070    0.0578
     XGB    0.9561     0.9444  0.9714 0.9577 0.9865 0.0500    0.1912
      NB    0.8293     0.8070  0.8762 0.8402 0.9043 0.1361    0.5061
     KNN    0.8634     0.8738  0.8571 0.8654 0.9629 0.0800    0.2170
Stacking    1.0000     1.0000  1.0000 1.0000 1.0000 0.0019 

## Evaluation on identical unseen-profile folds
Primary results give one vote per distinct profile. The three repeats have overlapping training sets; fold variation is not an independent-sample confidence interval.

In [4]:
scores = pd.read_csv(ROOT / 'results/fold_metrics.csv')
primary = scores[scores.weighting == 'unique']
print(primary.groupby('model')[['accuracy','precision','recall','f1','auc','brier']].mean().round(4).to_string())

                   accuracy  precision  recall      f1     auc   brier
model                                                                 
Baseline stacking    0.8192     0.8312  0.8457  0.8346  0.8960  0.1310
DT                   0.7529     0.7600  0.8009  0.7777  0.7483  0.2471
KNN                  0.7671     0.7768  0.8064  0.7896  0.8360  0.1795
LR                   0.8257     0.8299  0.8619  0.8416  0.9004  0.1278
NB                   0.8122     0.8208  0.8414  0.8277  0.8953  0.1455
Proposed pool        0.8488     0.8477  0.8843  0.8635  0.9150  0.1138
RF                   0.8180     0.8197  0.8559  0.8353  0.9005  0.1285
Typed stacking       0.8422     0.8438  0.8741  0.8565  0.9143  0.1155
Unique stacking      0.8313     0.8259  0.8782  0.8491  0.9088  0.1210
XGB                  0.8015     0.8098  0.8312  0.8183  0.8884  0.1414


![Mean accuracy and fold standard deviation](figures/accuracy_comparison.png)

![Brier score and fold standard deviation](figures/brier_comparison.png)

In [5]:
print('Row-weighted sensitivity:')
print(scores[scores.weighting == 'row'].groupby('model')[['accuracy','brier']].mean().round(4).to_string())
print('Learned pool weights averaged across folds:')
print(pd.read_csv(ROOT / 'results/pool_weights.csv')[['LR','DT','RF','XGB','NB','KNN']].mean().round(4).to_string())

Row-weighted sensitivity:
                   accuracy   brier
model                              
Baseline stacking    0.8160  0.1329
DT                   0.7506  0.2494
KNN                  0.7619  0.1840
LR                   0.8219  0.1295
NB                   0.8071  0.1483
Proposed pool        0.8453  0.1152
RF                   0.8165  0.1297
Typed stacking       0.8395  0.1167
Unique stacking      0.8268  0.1233
XGB                  0.7997  0.1419
Learned pool weights averaged across folds:
LR     0.4731
DT     0.0230
RF     0.1726
XGB    0.0508
NB     0.0667
KNN    0.2138


## Probability and error analysis
These plots use repeat 1 only, with one cross-fitted prediction per profile. Lower Brier loss does not alone prove better calibration.

![ROC and reliability](figures/roc_reliability.png)

![Confusion matrices](figures/confusion_matrices.png)

In [6]:
print((ROOT / 'results/uncertainty.json').read_text())
print((ROOT / 'results/verification.txt').read_text())

{
  "comparison": "proposed minus baseline Brier loss; negative favors proposed",
  "mean_profile_loss_difference": -0.01726225436856538,
  "conditional_bootstrap_95_interval": [
    -0.02743241267952476,
    -0.007551058274452643
  ],
  "resamples": 5000,
  "unit": "distinct profile, repeats kept together",
  "limitation": "Conditional on fitted cross-validated predictions; overlapping training sets and retraining uncertainty are not captured. Exploratory, not an independent-test significance claim."
}
PASS: data split profile disjointness; one outer prediction per profile per model per repeat; all 300 fold rows independently verified for accuracy, precision, recall, F1 and Brier; bounded probabilities. AUC uses scikit-learn. Full model fitting completed without errors.



## Explain these points in your own words
1. Why is the 100% random-row score misleading?
2. Why must preprocessing and stacking folds also respect training boundaries?
3. Which ablation made the largest difference?
4. What does Brier score measure, and why is it not a pure calibration metric?
5. Why does the bootstrap interval not prove performance on a new population?
6. What assumptions prevent this from being an exact replication of the paper?

Record a face-and-screen video after reviewing and understanding the work. Add the actual public-accessible links before submission.